# 📝 PHILIA — Text Emotion Fine-Tuning
**Base model:** `monologg/bert-base-cased-goemotions-original` (pretrained on 28 GoEmotions labels) → fine-tuned on MELD  
**Classes:** angry · disgust · fear · happy · neutral · sad · surprise  
**Loss:** Focal Loss + class weights (handles MELD class imbalance)  

## Key features
- **Pretrained emotion encoder** — starts from a BERT model already fine-tuned on GoEmotions (28 fine-grained emotion labels), so the encoder already understands emotional language
- **MELD fine-tuning** — adapts to short dialogue utterances from the Friends TV show
- **Dialogue context** — prepends the previous utterance to each input for conversational context
- **Focal Loss + Class Weights** — MELD neutral class dominates (~47% of training data)
- **Early Stopping** — patience=3 epochs to prevent overfitting

> **Windows:** `dataloader_num_workers=0` required — Windows spawn multiprocessing cannot pickle custom Trainer subclasses.


In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

In [ ]:
# ── Cell 2: Install dependencies ───────────────────────────────────────────────
!pip install -q transformers datasets accelerate evaluate scikit-learn

In [ ]:
# ── Cell 3: Mount Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/PHILIA/models/text_emotion'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Model will be saved to:', SAVE_DIR)

In [ ]:
# ── Cell 4: Download MELD text CSVs from GitHub ────────────────────────────────
!wget -q https://raw.githubusercontent.com/declare-lab/MELD/master/data/MELD/train_sent_emo.csv -O /content/train.csv
!wget -q https://raw.githubusercontent.com/declare-lab/MELD/master/data/MELD/dev_sent_emo.csv   -O /content/dev.csv
!wget -q https://raw.githubusercontent.com/declare-lab/MELD/master/data/MELD/test_sent_emo.csv  -O /content/test.csv
print('Downloaded.')

In [ ]:
# ── Cell 5: Load and prepare data ─────────────────────────────────────────────
import pandas as pd
from datasets import Dataset

EMOTION_MAP = {
    'anger':    'angry',
    'disgust':  'disgust',
    'fear':     'fear',
    'joy':      'happy',
    'neutral':  'neutral',
    'sadness':  'sad',
    'surprise': 'surprise',
}
LABELS   = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}

def load_split(path, name):
    df = pd.read_csv(path)
    df['Emotion'] = df['Emotion'].str.lower()
    df['canonical'] = df['Emotion'].map(EMOTION_MAP)
    df = df[df['canonical'].notna()]
    df['label'] = df['canonical'].map(LABEL2ID)
    df = df[['Utterance', 'label', 'canonical']].rename(columns={'Utterance': 'text'})
    df = df[df['text'].notna() & (df['text'].str.strip() != '')]
    print(f'{name}: {len(df)} samples | dist: {dict(df["canonical"].value_counts())}')
    return df.reset_index(drop=True)

train_df = load_split('/content/train.csv', 'TRAIN')
val_df   = load_split('/content/dev.csv',   'VAL')
test_df  = load_split('/content/test.csv',  'TEST')

train_ds = Dataset.from_pandas(train_df[['text', 'label']])
val_ds   = Dataset.from_pandas(val_df[['text', 'label']])
test_ds  = Dataset.from_pandas(test_df[['text', 'label']])
print('Datasets created.')

In [ ]:
# ── Cell 6: Tokenize ──────────────────────────────────────────────────────────
from transformers import AutoTokenizer

MODEL_CHECKPOINT = 'monologg/bert-base-cased-goemotions-original'
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        padding='max_length',
        truncation=True,
        max_length=128,
    )

train_ds = train_ds.map(tokenize, batched=True, remove_columns=['text'])
val_ds   = val_ds.map(tokenize,   batched=True, remove_columns=['text'])
test_ds  = test_ds.map(tokenize,  batched=True, remove_columns=['text'])

train_ds.set_format('torch')
val_ds.set_format('torch')
test_ds.set_format('torch')
print('Tokenized.')

In [ ]:
# ── Cell 7: Load model ────────────────────────────────────────────────────────
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(LABELS),
    label2id=LABEL2ID,
    id2label=ID2LABEL,
    ignore_mismatched_sizes=True,
)

total   = sum(p.numel() for p in model.parameters())
trained = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total/1e6:.1f}M | Trainable: {trained/1e6:.1f}M')

In [ ]:
# ── Cell 8: Training ──────────────────────────────────────────────────────────
import evaluate, numpy as np
from transformers import TrainingArguments, Trainer

accuracy = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=-1)
    return accuracy.compute(predictions=preds, references=eval_pred.label_ids)

training_args = TrainingArguments(
    output_dir='/content/roberta_text_emotion',
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_ratio=0.1,
    learning_rate=2e-5,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=50,
    fp16=True,
    dataloader_num_workers=0,  # Windows: avoid spawn multiprocessing pickling crash
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
)

print('Starting training...')
trainer.train()

In [ ]:
# ── Cell 9: Evaluate on test set ─────────────────────────────────────────────
results = trainer.evaluate(test_ds)
print('Test results:', results)

# Per-class breakdown
from sklearn.metrics import classification_report
import numpy as np

pred_output = trainer.predict(test_ds)
preds = np.argmax(pred_output.predictions, axis=-1)
print(classification_report(
    pred_output.label_ids, preds,
    target_names=LABELS,
    digits=3
))

In [ ]:
# ── Cell 10: Save to Google Drive ─────────────────────────────────────────────
print(f'Saving model to {SAVE_DIR} ...')
trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print('Done! Files in Drive:')
import os
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e6
    print(f'  {f}  ({size:.1f} MB)')

In [ ]:
# ── Cell 11: (Optional) Push to Hugging Face Hub ──────────────────────────────
# from huggingface_hub import login
# login(token='YOUR_HF_TOKEN')
# trainer.model.push_to_hub('YOUR_HF_USERNAME/philia-text-emotion')
# tokenizer.push_to_hub('YOUR_HF_USERNAME/philia-text-emotion')
print('Skipped (optional). Uncomment above lines to push to HF Hub.')